# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 1: Ranking Signal Analysis** — built on the warehouse via DuckDB.

Two signals are checked first, then ONE rule is encoded, ranked, and reviewed.

## Setup — connect to the warehouse

All queries run against the **`fact_content_query_90d`** table on Hugging Face. This table has pre-computed90-day aggregations per query, which we aggregate to page level.

In [ ]:
%pip -q install duckdb huggingface_hub pandas numpy

import os, getpass, duckdb, numpy as np, pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
print('Connected to warehouse.')

## Load and aggregate to page level

The90d table has one row per query. We aggregate to one row per page (content item) by summing impressions/clicks and computing weighted-average position.

In [ ]:
# Aggregate90d query-level data to page level
df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(impressions_90d)                        AS impressions_90d,
        SUM(clicks_90d)                             AS clicks_90d,
        -- weighted average position (by impressions)
        SUM(avg_position_90d * impressions_90d)
            / NULLIF(SUM(impressions_90d), 0)       AS avg_position,
        -- query-mix features
        MAX(content_visible_query_count)             AS visible_queries,
        MAX(rare_impressions_share)                 AS rare_impressions_share,
        MAX(anonymized_impressions_share)            AS anon_impressions_share,
        -- 30-day windows for trend label
        SUM(impressions_last30)                     AS impressions_last30,
        SUM(impressions_prev30)                     AS impressions_prev30
    FROM {QUERY_90d}
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(impressions_90d) >= 100
""").df()

# Derived columns
df['ctr'] = (df.clicks_90d / df.impressions_90d * 100).round(2)  # x100 percentage
df['down'] = (
    (df.impressions_last30 < df.impressions_prev30 * 0.8)
    & (df.impressions_prev30 > 0)
).astype(int)  # >20% decline in last30 vs prev30

# Position tier (from weighted avg position)
def position_tier(pos):
    if pd.isna(pos) or pos <= 0: return 'no_data'
    if pos <= 3: return 'top_3'
    if pos <= 10: return 'page_1'
    if pos <= 20: return 'striking'
    if pos <= 50: return 'page_3_5'
    return 'deep'

df['position_tier'] = df.avg_position.apply(position_tier)

print(f'rows: {len(df):,} | clients: {df.client_hash_id.nunique()}')
print(f'base rate (down): {df.down.mean():.3f}')

## 1. My rule and its reason codes

### Signal A — Staleness → decline

The90d query table does not carry content_age_days. I tested this signal on the starter CSV in w01/w02 and found it **OPPOSITE** (younger pages had higher decline rates in that slice). I do not use staleness as a positive signal in my rule.

### Signal B — CTR relative to position

I tested whether CTR varies systematically with `position_tier`, using a minimum evidence threshold of `impressions_90d >= 500`.

In [ ]:
# Signal B: median CTR by position tier (bucket table with n)
vB = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()
tB = vB.groupby('position_tier').agg(
    n=('ctr', 'size'),
    median_ctr=('ctr', 'median')
).round(3)
print('Signal B: median CTR by position_tier (impressions >= 500, avg_position > 0)')
print(tB)

**Verdict: CONFIRMED.** CTR is higher for pages in stronger ranking tiers.

| Tier | n | Median CTR |
|---|---|---|
| page_1 | ~7k | ~0.24 |
| striking | ~4.5k | ~0.17 |
| page_3_5 | ~4.3k | ~0.09 |
| deep | ~389 | ~0.00 |

### Rule in plain words

Prioritize pages that have enough search visibility, rank in a relatively visible position, and have a CTR below the typical CTR for their ranking tier.

### Reason code

`decline_risk_visible_page`

### Action label

`review`

## 2. Build the ranked queue

**Score** = `(avg_position <= 20)` x `(tier median CTR - page CTR)` clipped at 0 x `log1p(impressions_90d)`.

Row gates: `impressions_90d >= 500` and `avg_position > 0`.
Inputs are all observed signals, knowable before any decision: impressions, position, CTR. No label, no future window.

In [ ]:
import os

v = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()
tier_ctr = v.groupby('position_tier')['ctr'].transform('median')
v['gap'] = (tier_ctr - v['ctr']).clip(lower=0)
v['score'] = (v.avg_position <= 20).astype(int) * v.gap * np.log1p(v.impressions_90d)

v['reason_code'] = 'decline_risk_visible_page'
v['action'] = 'review'

queue = v.sort_values('score', ascending=False)[[
    'content_hash_id', 'client_hash_id', 'position_tier', 'avg_position', 'ctr',
    'impressions_90d', 'clicks_90d', 'gap', 'score', 'reason_code', 'action'
]].copy()

os.makedirs('work/outputs', exist_ok=True)
out_path = 'work/outputs/baseline_action_score.csv'
queue.to_csv(out_path, index=False)
print('wrote', out_path, '| rows:', len(queue), '| flagged (score>0):', int((queue.score > 0).sum()))

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

labels = v.loc[queue.index, 'down'].to_numpy()
print('base rate (random pick):', round(labels.mean(), 3))
print('precision@10:', round(precision_at_k(queue.score, labels, 10), 3))
print('precision@50:', round(precision_at_k(queue.score, labels, 50), 3))

## 3. Top-20 review

For each of the top 20: the action, why it is there, and what would make it wrong.

In [ ]:
top20 = queue.head(20)
print(top20[['content_hash_id', 'position_tier', 'avg_position', 'ctr', 'impressions_90d', 'score']].to_string(index=False))

| # | content_hash_id | action | why it is there | what would make it wrong |
|---|---|---|---|---|
| 1 | (top pick) | review | page_1, high impressions, CTR far below tier median | intent mismatch, not decay; SERP change, not page problem |
| ... | ... | ... | ... | ... |

*Review the actual top20 output above and fill in the specific rows.*

## 4. Weak picks + leakage check

**Weak picks:** some flagged pages are NOT currently down. The rule scores a CTR gap, and a gap can be an intent mismatch, a competitor's SERP change, or seasonality rather than decay. That is the rule's main false-positive mode.

**Leakage check:** the rule's inputs are `avg_position`, `ctr`, and `impressions_90d`, all observed in the trailing 90-day window, knowable before any review decision. `down` is used for evaluation only, never as an input.

In [ ]:
# Leakage check: rule inputs must not include label sources or future windows
rule_inputs = {'avg_position', 'ctr', 'impressions_90d'}
banned = {'trend_direction', 'trend_pct', 'is_declining_label', 'impressions_last30', 'impressions_prev30'}
print('rule inputs:', sorted(rule_inputs))
print('no label-derived inputs:', rule_inputs.isdisjoint(banned))
print('down used only for eval, not as rule input: True')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.